In [ ]:
import csv
import math
import random
import subprocess
from pathlib import Path

import pandas as pd


# ============================================================
# SETTINGS
# ============================================================

REPO_URL = "https://github.com/mggg/scot-elex.git"

# This is only a synchronized working copy of the GitHub repo.
REPO_DIR = Path.home() / ".cache" / "scot-elex"

OUTPUT_DIR = Path.home() / "Desktop" / "Scottish vote tables"

# Makes random tie-breaking reproducible.
RANDOM_SEED = 20260818

# Remove old vote-table CSVs before generating new ones.
CLEAR_OLD_OUTPUTS = True


# ============================================================
# YOUR SCOTTISH STV HELPERS
# ============================================================

def truncate(number, digits) -> float:
    stepper = 10.0 ** digits
    return math.trunc(stepper * number) / stepper


def update_rankings(row, hopefuls):
    """
    Remove candidates who are no longer hopeful and shift
    the remaining rankings left.
    """
    kept = [
        cand
        for cand in row.tolist()
        if cand != "skipped" and cand in hopefuls
    ]

    kept += ["skipped"] * (len(row) - len(kept))

    return kept


# ============================================================
# DOWNLOAD / UPDATE THE GITHUB REPOSITORY
# ============================================================

def sync_repo():
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)

    if not REPO_DIR.exists():
        print("Cloning scot-elex repository...")

        subprocess.run(
            [
                "git",
                "clone",
                REPO_URL,
                str(REPO_DIR)
            ],
            check=True
        )

    else:
        print("Updating scot-elex repository...")

        subprocess.run(
            [
                "git",
                "-C",
                str(REPO_DIR),
                "fetch",
                "origin"
            ],
            check=True
        )

        subprocess.run(
            [
                "git",
                "-C",
                str(REPO_DIR),
                "checkout",
                "main"
            ],
            check=True
        )

        subprocess.run(
            [
                "git",
                "-C",
                str(REPO_DIR),
                "reset",
                "--hard",
                "origin/main"
            ],
            check=True
        )

    print(f"Repository ready: {REPO_DIR}")


# ============================================================
# READ ONE REPOSITORY ELECTION
# ============================================================

def read_repo_election(file_path):
    """
    Converts one scot-elex CSV into the format expected by
    Scottish_STV:

        Count
        rank1
        rank2
        ...
        rankN

    Candidate names are preserved EXACTLY as they appear
    in the repository CSV.
    """

    with open(
        file_path,
        "r",
        newline="",
        encoding="utf-8-sig"
    ) as f:

        raw_rows = list(csv.reader(f))


    # --------------------------------------------------------
    # REMOVE COMPLETELY EMPTY ROWS, BUT DO NOT ALTER TEXT
    # --------------------------------------------------------

    rows = [
        row
        for row in raw_rows
        if any(cell.strip() != "" for cell in row)
    ]


    # --------------------------------------------------------
    # FIRST ROW:
    # number of candidates, number of seats
    # --------------------------------------------------------

    num_candidates = int(rows[0][0].strip())
    num_seats = int(rows[0][1].strip())


    # --------------------------------------------------------
    # FIND WHERE CANDIDATE INFORMATION BEGINS
    # --------------------------------------------------------

    candidate_start = None

    for i in range(1, len(rows)):

        if (
            len(rows[i]) > 0
            and rows[i][0].strip().startswith("Candidate ")
        ):

            candidate_start = i
            break


    if candidate_start is None:

        raise ValueError(
            f"Could not find candidate section in {file_path.name}"
        )


    # --------------------------------------------------------
    # BALLOT ROWS
    # --------------------------------------------------------

    ballot_rows = rows[1:candidate_start]


    # --------------------------------------------------------
    # CANDIDATE ROWS
    # --------------------------------------------------------

    candidate_rows = rows[
        candidate_start:
        candidate_start + num_candidates
    ]


    if len(candidate_rows) != num_candidates:

        raise ValueError(
            f"{file_path.name}: expected {num_candidates} "
            f"candidate rows but found {len(candidate_rows)}"
        )


    # --------------------------------------------------------
    # CANDIDATE NUMBER -> EXACT REPO NAME
    # --------------------------------------------------------

    candidate_names = {}


    for row in candidate_rows:

        candidate_number = int(
            row[0]
            .replace("Candidate", "")
            .strip()
        )

        # IMPORTANT:
        # Do NOT strip or otherwise modify candidate name.
        candidate_name = row[1]

        candidate_names[
            candidate_number
        ] = candidate_name


    # --------------------------------------------------------
    # CHECK CANDIDATE NUMBERING
    # --------------------------------------------------------

    expected_numbers = set(
        range(1, num_candidates + 1)
    )

    actual_numbers = set(
        candidate_names.keys()
    )


    if actual_numbers != expected_numbers:

        raise ValueError(
            f"{file_path.name}: candidate numbering does not "
            f"match 1 through {num_candidates}"
        )


    # Exact order and exact names from repo.
    candidate_order = [
        candidate_names[i]
        for i in range(
            1,
            num_candidates + 1
        )
    ]


    if (
        len(candidate_order)
        != len(set(candidate_order))
    ):

        raise ValueError(
            f"{file_path.name}: duplicate candidate names found."
        )


    # --------------------------------------------------------
    # CREATE rank1, rank2, ..., rankN
    # --------------------------------------------------------

    rank_columns = [
        f"rank{i}"
        for i in range(
            1,
            num_candidates + 1
        )
    ]

    profile_data = []


    for row in ballot_rows:

        # Ignore blank cells at end of ballot row.
        usable_cells = [
            cell
            for cell in row
            if cell.strip() != ""
        ]


        if len(usable_cells) == 0:
            continue


        ballot_count = float(
            usable_cells[0].strip()
        )


        numbered_preferences = [
            int(x.strip())
            for x in usable_cells[1:]
        ]


        # Validate candidate numbers.
        for candidate_number in numbered_preferences:

            if candidate_number not in candidate_names:

                raise ValueError(
                    f"{file_path.name}: ballot contains "
                    f"candidate number {candidate_number}, "
                    f"which is not in candidate list."
                )


        # Replace numbers with EXACT candidate strings.
        preferences = [
            candidate_names[x]
            for x in numbered_preferences
        ]


        # Pad partial rankings.
        preferences += (
            ["skipped"]
            * (
                num_candidates
                - len(preferences)
            )
        )


        profile_data.append(
            [ballot_count]
            + preferences
        )


    frame = pd.DataFrame(
        profile_data,
        columns=[
            "Count"
        ] + rank_columns
    )


    return (
        frame,
        num_candidates,
        num_seats,
        candidate_order
    )


# ============================================================
# SCOTTISH STV WITH VOTE TOTALS RECORDED BY ROUND
# ============================================================

def Scottish_STV_votes_by_round(frame, n, S):

    winners = []
    hopefuls = []

    rank_columns = [
        col
        for col in frame.columns
        if col.startswith("rank")
    ]

    cands = pd.unique(
        frame[rank_columns].values.ravel()
    ).tolist()

    cands = [
        cand
        for cand in cands
        if cand != "skipped"
    ]

    n = len(cands)

    for cand in cands:
        hopefuls.append(cand)

    frame = frame.copy(deep=True)

    # --------------------------------------------------------
    # IMPORTANT:
    #
    # Keep the NUMBER OF BALLOTS in a parcel separate from the
    # CURRENT TRANSFER VALUE of those ballots.
    #
    # This is necessary because Scottish STV calculates a new
    # transfer value as
    #
    #   truncate(
    #       surplus * current_transfer_value / candidate_total,
    #       5
    #   )
    #
    # rather than truncating surplus/candidate_total first and
    # then multiplying by the current transfer value.
    # --------------------------------------------------------

    frame["BallotCount"] = frame["Count"].astype(float)
    frame["TransferValue"] = 1.0
    frame = frame.drop(columns=["Count"])

    original_ballot_total = frame["BallotCount"].sum()

    quota = (
        math.floor(
            original_ballot_total / (S + 1)
        )
        + 1
    )

    # --------------------------------------------------------
    # HELPER: GROUP IDENTICAL PARCELS
    #
    # Ballots can be combined only when they have BOTH:
    #   1. the same remaining rankings, and
    #   2. the same current transfer value.
    # --------------------------------------------------------

    def group_parcels(current_frame):

        current_frame = (
            current_frame.groupby(
                rank_columns + ["TransferValue"],
                as_index=False,
                dropna=False
            )["BallotCount"].sum()
        )

        current_frame = current_frame[
            ~current_frame[
                rank_columns
            ].eq(
                "skipped"
            ).all(axis=1)
        ]

        return current_frame.reset_index(drop=True)

    # --------------------------------------------------------
    # HELPER: RECOUNT
    #
    # A parcel contributes:
    #
    #   number of ballots * transfer value
    # --------------------------------------------------------

    def recount():

        new_vote_counts = {
            cand: 0.0
            for cand in cands
        }

        for k in range(len(frame)):

            first_choice = frame.at[k, "rank1"]

            if first_choice in new_vote_counts:

                new_vote_counts[first_choice] += (
                    frame.at[k, "BallotCount"]
                    * frame.at[k, "TransferValue"]
                )

        return new_vote_counts

    # --------------------------------------------------------
    # INITIAL VOTE TOTALS
    # --------------------------------------------------------

    vote_counts = recount()

    # --------------------------------------------------------
    # STORAGE FOR ROUND TABLE
    # --------------------------------------------------------

    rounds = []

    eliminated = set()

    # Candidates whose surplus has been transferred.
    transferred = set()

    def record_round():

        snapshot = {}

        for cand in cands:

            if cand in eliminated:

                snapshot[cand] = 0.0

            elif cand in transferred:

                # Once the surplus has been transferred,
                # display the elected candidate at quota.
                snapshot[cand] = float(quota)

            else:

                snapshot[cand] = float(
                    vote_counts.get(cand, 0.0)
                )

        # Avoid recording an identical round twice.
        if len(rounds) == 0:

            rounds.append(snapshot)

        else:

            different = any(
                abs(
                    snapshot[cand]
                    - rounds[-1][cand]
                ) > 1e-12
                for cand in cands
            )

            if different:
                rounds.append(snapshot)

    # Round 1 = original first-choice totals.
    record_round()

    # ========================================================
    # MAIN STV LOOP
    # ========================================================

    while len(winners) < S:

        newly_elected = {
            cand: vote_counts[cand]
            for cand in vote_counts
            if (
                vote_counts[cand] >= quota
                and cand not in winners
            )
        }

        newly_elected = dict(
            sorted(
                newly_elected.items(),
                key=lambda x: x[1],
                reverse=True
            )
        )

        # ====================================================
        # SOMEONE HAS REACHED QUOTA
        # ====================================================

        if len(newly_elected) > 0:

            simultaneous_winners = list(
                newly_elected.keys()
            )

            for cand in simultaneous_winners:

                if cand not in winners:
                    winners.append(cand)

                if cand in hopefuls:
                    hopefuls.remove(cand)

            # If enough candidates have already been elected,
            # the election is complete.
            if len(winners) >= S:
                break

            surplus_queue = (
                simultaneous_winners.copy()
            )

            already_transferred = []

            # ------------------------------------------------
            # TRANSFER SURPLUSES
            # ------------------------------------------------

            while len(surplus_queue) > 0:

                elected_cand = (
                    surplus_queue.pop(0)
                )

                elected_total = vote_counts.get(
                    elected_cand,
                    0.0
                )

                # No surplus to transfer.
                #
                # If the candidate is exactly at quota, ballots
                # currently allocated to that candidate are set
                # aside rather than transferred. The elected
                # candidate is also removed from lower rankings
                # on the remaining parcels.
                if elected_total <= quota:

                    keep_rows = []

                    for k in range(len(frame)):

                        current_first = (
                            frame.at[k, "rank1"]
                        )

                        if current_first == elected_cand:
                            continue

                        ranks = [
                            frame.at[k, col]
                            for col in rank_columns
                        ]

                        new_ranks = []

                        for r in ranks:

                            if r == "skipped":
                                continue

                            if r == elected_cand:
                                continue

                            if r in already_transferred:
                                continue

                            if r in surplus_queue:

                                if current_first == r:
                                    new_ranks.append(r)

                                continue

                            new_ranks.append(r)

                        new_ranks += (
                            ["skipped"]
                            * (
                                len(ranks)
                                - len(new_ranks)
                            )
                        )

                        keep_rows.append(
                            new_ranks
                            + [
                                frame.at[k, "BallotCount"],
                                frame.at[k, "TransferValue"]
                            ]
                        )

                    frame = pd.DataFrame(
                        keep_rows,
                        columns=(
                            rank_columns
                            + [
                                "BallotCount",
                                "TransferValue"
                            ]
                        )
                    )

                    frame = group_parcels(frame)

                    already_transferred.append(
                        elected_cand
                    )

                    transferred.add(
                        elected_cand
                    )

                    vote_counts = recount()

                    record_round()

                    continue

                surplus = (
                    elected_total
                    - quota
                )

                # --------------------------------------------
                # REWEIGHT BALLOTS FOR ELECTED CANDIDATE
                #
                # OFFICIAL SCOTTISH ORDER:
                #
                # new TV =
                # truncate(
                #     surplus * current TV / elected total,
                #     5
                # )
                #
                # The truncation therefore occurs AFTER the
                # existing transfer value has been included.
                # --------------------------------------------

                for k in range(len(frame)):

                    ranks = [
                        frame.at[k, col]
                        for col in rank_columns
                    ]

                    current_first = (
                        frame.at[k, "rank1"]
                    )

                    if (
                        current_first
                        == elected_cand
                    ):

                        current_value = (
                            frame.at[
                                k,
                                "TransferValue"
                            ]
                        )

                        frame.at[
                            k,
                            "TransferValue"
                        ] = truncate(
                            (
                                surplus
                                * current_value
                            )
                            / elected_total,
                            5
                        )

                    new_ranks = []

                    for r in ranks:

                        if r == "skipped":
                            continue

                        if r == elected_cand:
                            continue

                        if r in already_transferred:
                            continue

                        if r in surplus_queue:

                            if current_first == r:
                                new_ranks.append(r)

                            continue

                        new_ranks.append(r)

                    new_ranks += (
                        ["skipped"]
                        * (
                            len(ranks)
                            - len(new_ranks)
                        )
                    )

                    for col, new_rank in zip(
                        rank_columns,
                        new_ranks
                    ):

                        frame.at[
                            k,
                            col
                        ] = new_rank

                # Keep parcels with different transfer values
                # separate, even if their remaining rankings
                # are identical.
                frame = group_parcels(frame)

                already_transferred.append(
                    elected_cand
                )

                transferred.add(
                    elected_cand
                )

                vote_counts = recount()

                # Record the new tally after surplus transfer.
                record_round()

                # --------------------------------------------
                # DID THE TRANSFER ELECT SOMEONE ELSE?
                # --------------------------------------------

                additional_elected = {
                    cand: vote_counts[cand]
                    for cand in vote_counts
                    if (
                        vote_counts[cand] >= quota
                        and cand not in winners
                    )
                }

                additional_elected = dict(
                    sorted(
                        additional_elected.items(),
                        key=lambda x: x[1],
                        reverse=True
                    )
                )

                for cand in additional_elected:

                    if cand not in winners:
                        winners.append(cand)

                    if cand in hopefuls:
                        hopefuls.remove(cand)

                    if (
                        cand not in surplus_queue
                        and cand
                        not in already_transferred
                    ):

                        surplus_queue.append(cand)

                if len(winners) >= S:
                    break

            if len(winners) >= S:
                break

        # ====================================================
        # REMAINING HOPEFULS EXACTLY FILL REMAINING SEATS
        # ====================================================

        elif (
            len(hopefuls)
            + len(winners)
            == S
        ):

            winners = (
                winners
                + hopefuls
            )

            break

        # ====================================================
        # ELIMINATION
        # ====================================================

        else:

            positive_votes = [
                vote_counts[cand]
                for cand in hopefuls
                if vote_counts.get(cand, 0.0) > 0
            ]

            if len(positive_votes) == 0:

                winners = (
                    winners
                    + hopefuls
                )

                break

            min_count = min(
                positive_votes
            )

            lowest_hopefuls = [
                cand
                for cand in hopefuls
                if vote_counts.get(cand, 0.0) == min_count
            ]

            count = len(lowest_hopefuls)

            # ------------------------------------------------
            # ONE LOWEST CANDIDATE
            # ------------------------------------------------

            if count == 1:

                eliminated_cand = (
                    lowest_hopefuls[0]
                )

                if (
                    eliminated_cand
                    in hopefuls
                ):

                    hopefuls.remove(
                        eliminated_cand
                    )

                eliminated.add(
                    eliminated_cand
                )

                # Elimination transfers do NOT change the
                # transfer value of a parcel. They simply remove
                # the eliminated candidate from the rankings.
                updated_ranks = (
                    frame[
                        rank_columns
                    ].apply(
                        lambda row:
                        update_rankings(
                            row,
                            hopefuls
                        ),
                        axis=1
                    )
                )

                new_profile = pd.DataFrame(
                    updated_ranks.tolist(),
                    columns=rank_columns
                )

                new_profile["BallotCount"] = (
                    frame["BallotCount"].values
                )

                new_profile["TransferValue"] = (
                    frame["TransferValue"].values
                )

                frame = group_parcels(
                    new_profile
                )

                vote_counts = recount()

                # Record tally after elimination transfer.
                record_round()

                if (
                    len(hopefuls)
                    + len(winners)
                    == S
                ):

                    winners = (
                        winners
                        + hopefuls
                    )

                    break

            # ------------------------------------------------
            # MULTIPLE CANDIDATES TIED AT BOTTOM
            # ------------------------------------------------

            elif (
                len(hopefuls)
                - count
                >= S - len(winners)
            ):

                eliminated_cands = (
                    lowest_hopefuls.copy()
                )

                # This matches your existing function.
                random.shuffle(
                    eliminated_cands
                )

                for eliminated_cand in (
                    eliminated_cands
                ):

                    if (
                        eliminated_cand
                        in hopefuls
                    ):

                        hopefuls.remove(
                            eliminated_cand
                        )

                    eliminated.add(
                        eliminated_cand
                    )

                    updated_ranks = (
                        frame[
                            rank_columns
                        ].apply(
                            lambda row:
                            update_rankings(
                                row,
                                hopefuls
                            ),
                            axis=1
                        )
                    )

                    new_profile = pd.DataFrame(
                        updated_ranks.tolist(),
                        columns=rank_columns
                    )

                    new_profile["BallotCount"] = (
                        frame["BallotCount"].values
                    )

                    new_profile["TransferValue"] = (
                        frame["TransferValue"].values
                    )

                    frame = group_parcels(
                        new_profile
                    )

                    if (
                        len(hopefuls)
                        + len(winners)
                        == S
                    ):

                        break

                vote_counts = recount()

                # Treat the tied elimination group as one
                # tally transition, matching your function.
                record_round()

                if (
                    len(hopefuls)
                    + len(winners)
                    == S
                ):

                    winners = (
                        winners
                        + hopefuls
                    )

                    break

            else:

                winners = (
                    winners
                    + hopefuls
                )

                break

    # ========================================================
    # CREATE OUTPUT TABLE
    # ========================================================

    vote_table = pd.DataFrame(
        index=cands
    )

    vote_table.index.name = "Candidate"

    for round_number, snapshot in enumerate(
        rounds,
        start=1
    ):

        vote_table[
            f"Round {round_number}"
        ] = [
            snapshot[cand]
            for cand in cands
        ]

    return (
        winners[:S],
        quota,
        vote_table
    )


# ============================================================
# PROCESS EVERY ELECTION IN EVERY *_cands FOLDER
# ============================================================

def process_all_elections():

    random.seed(RANDOM_SEED)

    sync_repo()


    # --------------------------------------------------------
    # CREATE OUTPUT DIRECTORY
    # --------------------------------------------------------

    OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True
    )


    if CLEAR_OLD_OUTPUTS:

        for old_file in OUTPUT_DIR.glob(
            "*_votes_table.csv"
        ):

            old_file.unlink()


    # --------------------------------------------------------
    # FIND ONLY FOLDERS ENDING IN _cands
    # --------------------------------------------------------

    candidate_folders = sorted(
        [
            folder
            for folder in REPO_DIR.iterdir()
            if (
                folder.is_dir()
                and folder.name.endswith(
                    "_cands"
                )
            )
        ],
        key=lambda x: int(
            x.name.split("_")[0]
        )
    )


    print()
    print("Candidate folders found:")

    for folder in candidate_folders:
        print(f"  {folder.name}")


    # --------------------------------------------------------
    # GET EVERY ELECTION CSV
    # --------------------------------------------------------

    election_files = []

    for folder in candidate_folders:

        election_files.extend(
            sorted(
                folder.glob("*.csv")
            )
        )


    print()
    print(
        f"Total election files found: "
        f"{len(election_files)}"
    )
    print()


    # --------------------------------------------------------
    # CHECK FOR OUTPUT-FILENAME COLLISIONS
    # --------------------------------------------------------

    output_names = [
        (
            f"{file_path.stem}"
            f"_votes_table.csv"
        )
        for file_path in election_files
    ]


    if (
        len(output_names)
        != len(set(output_names))
    ):

        duplicates = sorted(
            {
                name
                for name in output_names
                if output_names.count(name) > 1
            }
        )

        raise ValueError(
            "Duplicate output filenames would be "
            f"created:\n{duplicates}"
        )


    # --------------------------------------------------------
    # PROCESS
    # --------------------------------------------------------

    successes = 0
    failures = []


    for i, file_path in enumerate(
        election_files,
        start=1
    ):

        try:

            (
                frame,
                n,
                S,
                candidate_order
            ) = read_repo_election(
                file_path
            )


            winners, quota, vote_table = (
                Scottish_STV_votes_by_round(
                    frame,
                    n,
                    S
                )
            )


            # Put candidates in the exact Candidate 1,
            # Candidate 2, ... order from the repo.
            #
            # fillna(0) also guarantees that a candidate
            # appearing in the repo candidate list is not
            # accidentally omitted from the output.
            vote_table = (
                vote_table
                .reindex(candidate_order)
                .fillna(0)
            )


            output_filename = (
                f"{file_path.stem}"
                f"_votes_table.csv"
            )


            output_path = (
                OUTPUT_DIR
                / output_filename
            )


            vote_table.to_csv(
                output_path
            )


            successes += 1


        except Exception as e:

            failures.append(
                {
                    "file": str(file_path),
                    "error": str(e)
                }
            )


        if (
            i % 50 == 0
            or i == len(election_files)
        ):

            print(
                f"Processed "
                f"{i:,} / "
                f"{len(election_files):,}"
            )


    # --------------------------------------------------------
    # SUMMARY
    # --------------------------------------------------------

    print()
    print("=" * 60)
    print("FINISHED")
    print("=" * 60)

    print(
        f"Successful elections: "
        f"{successes:,}"
    )

    print(
        f"Failed elections: "
        f"{len(failures):,}"
    )

    print(
        f"Output folder: "
        f"{OUTPUT_DIR}"
    )


    if failures:

        error_path = (
            OUTPUT_DIR
            / "_processing_errors.csv"
        )

        pd.DataFrame(
            failures
        ).to_csv(
            error_path,
            index=False
        )

        print()
        print(
            "Errors were written to:"
        )

        print(error_path)

        raise RuntimeError(
            f"{len(failures)} election(s) "
            f"failed. Check "
            f"{error_path.name}."
        )

    else:

        error_path = (
            OUTPUT_DIR
            / "_processing_errors.csv"
        )

        if error_path.exists():
            error_path.unlink()

        print()
        print(
            "All elections processed "
            "successfully."
        )


# ============================================================
# RUN EVERYTHING
# ============================================================

process_all_elections()